# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Nguyễn Đăng Thực — MHV 2A202603014

Notebook chạy trên Kaggle (GPU T4). Code nằm trong các file `.py` cùng thư mục (`dataset.py`, `model.py`,
`losses.py`, `train.py`, `inference.py`, `benchmark.py`, `experiments.py`, `make_results.py`), notebook chỉ gọi hàm.
Mọi lựa chọn đều làm trên **val**; test chỉ chạy ở Bước 4, mỗi seed một lần.

Biến môi trường `LAB_SMOKE=1` thì chạy bản rút gọn (1 epoch, 24 ảnh train mỗi lớp) để kiểm tra code trước khi chạy thật.

## 0. Cài đặt môi trường

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "openpyxl", "fvcore"], check=False)

import os, platform, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch, timm, torchvision
from IPython.display import Image as ShowImage, display, Markdown

CODE_DIR = Path(os.environ.get("LAB_CODE_DIR", ".")).resolve()
sys.path.insert(0, str(CODE_DIR))
SMOKE = os.environ.get("LAB_SMOKE") == "1"
OUT = Path(os.environ.get("LAB_OUT", "out")).resolve()
CKPT = os.environ.get("LAB_CKPT", str(OUT / "ckpt"))
DATA = Path(os.environ.get("LAB_DATA", "data")).resolve()
OUT.mkdir(parents=True, exist_ok=True)

print("python", platform.python_version(), "| torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| timm", timm.__version__, "| pandas", pd.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
print("SMOKE =", SMOKE, "| OUT =", OUT)

### Tải dữ liệu
Ảnh từ Zenodo (kiểm tra MD5), nhãn và file chia fold nguyên bản từ GitHub của tác giả.

In [ ]:
import hashlib, urllib.request, zipfile

(DATA / "labels").mkdir(parents=True, exist_ok=True)
zip_path = DATA / "images.zip"
if not zip_path.exists():
    urllib.request.urlretrieve("https://zenodo.org/records/7939060/files/images.zip?download=1", zip_path)
h = hashlib.md5()
with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
assert h.hexdigest() == "b7b30f96d466fba86016aa5a26606e0f", f"MD5 sai: {h.hexdigest()}"
print("MD5 đúng")

if not any(DATA.rglob("*.jpg")):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA / "images_raw")
BASE = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    p = DATA / "labels" / f"{name}.csv"
    if not p.exists():
        urllib.request.urlretrieve(f"{BASE}/{name}.csv", p)

# thư mục thật sự chứa ảnh .jpg
jpg_dirs = pd.Series([str(p.parent) for p in DATA.rglob("*.jpg")]).value_counts()
print(jpg_dirs)
IMAGES_DIR = jpg_dirs.index[0]
LABELS_DIR = str(DATA / "labels")
print("IMAGES_DIR =", IMAGES_DIR, "| số file:", len(os.listdir(IMAGES_DIR)))

## Bước 0 — EDA và kiểm tra pipeline
### 0.1 Kiểm tra split (README mục 2.1)

In [ ]:
from dataset import load_split, check_split, preload_images, image_modes, CLASS_NAMES
train_df, val_df, test_df = load_split(LABELS_DIR, fold=0)
split_info = check_split(train_df, val_df, test_df, IMAGES_DIR)
json.dump({k: v for k, v in split_info.items() if k != "per_class"}, open(OUT / "split_check.json", "w"),
          ensure_ascii=False, indent=2)
split_info["per_class"].to_csv(OUT / "split_per_class.csv")
display(split_info["per_class"])

### 0.2 EDA: phân bố lớp, ảnh mẫu, kích thước và số kênh

In [ ]:
from pipeline_checks import eda_plots
FIGS = OUT / "figures"
eda = eda_plots(split_info, train_df, IMAGES_DIR, FIGS)
print(eda)

all_files = pd.concat([train_df, val_df, test_df])["Filename"].tolist()
t0 = time.time()
shapes = preload_images(all_files, IMAGES_DIR)   # giải mã một lần, giữ trong RAM cho mọi lần chạy
print(f"nạp {len(all_files)} ảnh vào RAM trong {time.time() - t0:.0f}s; kích thước (H, W, C):", shapes)
print("chế độ màu gốc (2000 ảnh đầu):", image_modes(all_files[:2000], IMAGES_DIR))
json.dump({"eda": eda, "shapes": shapes}, open(OUT / "eda.json", "w"), ensure_ascii=False, indent=2, default=str)
display(ShowImage(str(FIGS / "eda_class_distribution.png")))
display(ShowImage(str(FIGS / "eda_samples.png")))

### 0.3 Kiểm tra pipeline: seed, loss ban đầu, overfit một batch nhỏ, ảnh sau augmentation, train/eval mode

In [ ]:
from train import set_seed
from pipeline_checks import initial_loss, overfit_small_batch, show_augmented, check_modes
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(0)
checks = {}
checks["initial_loss"] = [initial_loss(b, val_df, IMAGES_DIR, device)
                          for b in ("resnet50", "convnext_tiny", "deit_small_patch16_224")]
checks["overfit"] = overfit_small_batch("resnet50", train_df, IMAGES_DIR, device, n=32,
                                        steps=40 if SMOKE else 100, fig_path=FIGS / "check_overfit_32.png")
show_augmented(train_df, IMAGES_DIR, FIGS / "check_aug_basic.png", "basic")
show_augmented(train_df, IMAGES_DIR, FIGS / "check_aug_trivial.png", "trivial")
checks["modes"] = check_modes(device)
json.dump(checks, open(OUT / "pipeline_checks.json", "w"), ensure_ascii=False, indent=2)
print(json.dumps(checks, indent=2, ensure_ascii=False))
for f in ("check_overfit_32.png", "check_aug_basic.png", "check_aug_trivial.png"):
    display(ShowImage(str(FIGS / f)))

### 0.4 Test tự viết (focal γ=0, CutMix, gộp BN, ...) và test của repo

In [ ]:
r = subprocess.run([sys.executable, "-m", "unittest", "test_own", "-v"], cwd=CODE_DIR, capture_output=True, text=True)
print(r.stderr[-6000:])
(OUT / "test_own.txt").write_text(r.stderr)
assert r.returncode == 0, "test tự viết lỗi"
repo = CODE_DIR.parent
if (repo / "tests").exists():
    r = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"], cwd=repo, capture_output=True, text=True)
    print(r.stderr[-3000:])
    (OUT / "test_repo.txt").write_text(r.stderr)

## Bước 1 — So sánh backbone (5 backbone, công thức nền T00, seed 0)
ResNet-50 (mốc), ConvNeXt-T, DeiT-S, EfficientNet-B0, MobileNetV3-L. Cùng công thức: AdamW, LR backbone 1e-4 / head 1e-3,
wd 0,05 (không áp cho norm/bias), warmup + cosine theo bước, AMP, CE, chọn checkpoint theo macro-F1 val.

**Giảm do ngân sách GPU** (cả notebook phải chạy trong khoảng 30 phút trên một T4): ảnh train 128×128
(RandomResizedCrop 128), 4 epoch, batch 128, warmup 0,5 epoch. Val/test: resize 146 rồi center crop 128.
DeiT-S dùng trọng số 224 được timm nội suy pos-embed về 128. Mọi backbone vẫn cùng một công thức.

In [ ]:
from experiments import Lab, step1, step2, step3, step4, run_eval, confusion_and_errors, BACKBONES
lab = Lab(images_dir=IMAGES_DIR, labels_dir=LABELS_DIR, out=str(OUT), ckpt_dir=CKPT, num_workers=4, smoke=SMOKE)
df1 = step1(lab)
display(df1[["exp_id", "backbone", "weight_tag", "params_m", "gmacs", "best_epoch", "val_macro_f1", "val_top1",
             "sec_per_epoch"]])
dec = lab.load_json("decisions.json")
print("Chọn đi tiếp:", dec["step1_backbone"])
for e, _, d in BACKBONES:
    display(ShowImage(str(OUT / "curves" / f"{e}_{d}.png")))

## Bước 2 — Công thức huấn luyện (backbone đã chọn ở Bước 1)
T00 chạy 3 seed để đo nhiễu (seed 0 chính là lần chạy B0x của backbone này, chép lại, không train lại).
Mỗi T01–T06 chỉ khác T00 một yếu tố (trục A, B, C, F). T07 là kết hợp các yếu tố tốt
(tham lam theo trục: chỉ lấy yếu tố có Δ > std nhiễu; xem `decisions.json`).

In [ ]:
df2 = step2(lab, dec["step1_backbone"]["backbone"])
dec = lab.load_json("decisions.json")
display(df2[["exp_id", "seed", "axis", "change", "best_epoch", "val_macro_f1", "val_top1", "delta_vs_T00",
             "noise_std", "distinguishable"]])
print(json.dumps({k: dec[k] for k in ("step2_noise_std", "step2_combination", "step2_recipe")}, indent=2, ensure_ascii=False))
for p in sorted((OUT / "curves").glob("T*.png")):
    display(ShowImage(str(p)))

## Bước 3 — Suy luận và độ trễ (không train lại)
Mô hình: cấu hình tốt nhất của Bước 2 (seed 0). Mọi số trên **val**. Độ trễ: warmup 10 lần, `torch.cuda.synchronize()`
trước và sau, 60 lần đo, p50/p95/p99, batch 1 và batch 32; mặc định không tính tiền xử lý (có một dòng tính cả tiền xử lý).

In [ ]:
res3 = step3(lab)
dec = lab.load_json("decisions.json")
pd.set_option("display.max_colwidth", 60)
display(res3["inference"])
display(res3["latency"][["config", "dtype", "batch", "img_size", "fused_bn", "preprocess", "p50", "p95", "p99", "images_per_s"]])
print(json.dumps(dec["step3_method"], indent=2, ensure_ascii=False))
display(ShowImage(str(OUT / "figures" / "accuracy_latency_tradeoff.png")))
display(ShowImage(str(OUT / "figures" / "reliability_val.png")))

## Bước 4 — Chung kết: 3 seed, test một lần mỗi seed
F01 = backbone + công thức + cách suy luận đã chốt trên val, rồi temperature scaling (T khớp trên val của từng seed).
Mốc = T00 + I00, dùng lại checkpoint T00 seed 0/1/2 đã train ở Bước 2 (chọn trên val), chạy test một lần.

In [ ]:
res4 = step4(lab)
display(res4["final_val"])
inf_df = res3["inference"].set_index("exp_id")
p95 = float(inf_df.loc[dec["step3_method"]["method"] if dec["step3_method"]["method"] in inf_df.index
                       else "I04_" + dec["step3_method"]["method"], "lat_b1_p95_ms"])
print("p95 batch 1 của cấu hình chung kết:", p95, "ms")
logs = run_eval(lab, latency_p95_ms=p95)
confusion_and_errors(lab)
for f in ("confusion_test_F01.png", "confusion_test_T00.png", "errors_hard_classes.png"):
    if (OUT / "figures" / f).exists():
        display(ShowImage(str(OUT / "figures" / f)))

## Phần làm thêm
1. Lệch phân phối tự tạo trên **val** (mờ, tối, nhiễu, JPEG nén mạnh): macro-F1 và ECE trước/sau temperature scaling
   (T khớp trên val sạch). Không dùng test.
2. Grad-CAM trên các ảnh test đã bị đoán sai của Chinee apple / Snake weed (chỉ để giải thích lỗi, không quyết định gì).

In [ ]:
from bonus import shift_eval, gradcam_errors
display(shift_eval(lab))
g = gradcam_errors(lab)
if g:
    display(ShowImage(str(g)))

## Bước 5 — results.xlsx, biểu đồ tổng hợp, bảng cho báo cáo

In [ ]:
import make_results
make_results.main(str(OUT))
xl = pd.read_excel(OUT / "results.xlsx", sheet_name=None)
for name, df in xl.items():
    print("=====", name, df.shape)
    display(df.head(15))
display(ShowImage(str(OUT / "figures" / "backbones_f1_latency_params.png")))